# Beamtime on ODIN: starter notebook

Your proposal is in `beamtime.md`. This notebook only gives you the tools; what to measure and how to analyze it is
up to you. Keep notes of your choices: they are part of your report.

**Tools**
- `rows`: your logbook (one line per count: file, title, settings);
- `red.histograms(file, LAM, ROI)`: image and spectrum of one file, λ from the ESS reduction;
- `red.reduce(sample, open_beam, dark=..., lam=LAM, roi=ROI)`: transmission image and spectrum (official ESS code);
- `curve_fit` (scipy) for fits, `erfc` for smoothed steps; `h5py` to look inside a NeXus file.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))     # the modules of the ODIN twin (folder odin/)

In [ ]:
from odin_paths import TWIN_RUN_ROOT
FOLDER = f"{TWIN_RUN_ROOT}/data/odin/2026/1234/raw"     # the folder of your proposal (NICOS)

In [ ]:
import sys, os, json, glob, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import scipp as sc
import h5py
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.special import erfc
from odin_paths import TWIN_RUN_ROOT
import odin_reduction as red

In [ ]:
rows = []
with open(f"{FOLDER}/log.jsonl", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        if entry["action"] != "count" or not entry["details"].get("file"):
            continue
        d, s = entry["details"], entry["details"]["state"]
        kind = "dark" if s["shutter"] == "closed" else ("open beam" if abs(s["stage_x"]) > 0.02 else "sample")
        rows.append(dict(file=os.path.basename(d["file"]), kind=kind, title=d["title"], t_s=d["t"],
                         counts=d["n_counts"], choppers=s["choppers"], wfm_delta_m=s["wfm_delta"],
                         pinhole_m=s["pinhole"], stage_x_m=s["stage_x"], stage_rot_deg=s["stage_rot"]))
def show_table(rows):                                  # a list of dicts, printed as a table
    if not rows:
        print("no measurement in the logbook"); return
    cols = list(rows[0])
    width = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in cols}
    print("  ".join(c.ljust(width[c]) for c in cols))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(width[c]) for c in cols))

show_table(rows)

## Your analysis

## Results for the committee

(edge position, $d_{110}$, $a$, with uncertainties; logbook of your choices)

## Check your results (after writing them)

The twin knows the true sample: put your final numbers below and run the cell. A wrong result is explained.

In [ ]:
import odin_exercises as ex
lam_edge, dlam_edge = None, None     # your (110) edge and its uncertainty [Å]
a_fe, da_fe = None, None             # your lattice parameter and its uncertainty [Å]
ex.check_edge(lam_edge, dlam_edge)
ex.check_lattice(a_fe, da_fe)